## AI Without the PhD — Cortex AI
Most teams think AI on their data requires a data science team and a PhD. This lab uses three managed Cortex AI functions, a first-use-case exercise, Snowflake CoWork, and Snowflake CoCo to make practical AI workflows approachable from SQL and natural language.
> **Edition:** Snowflake CoWork and Snowflake CoCo are concept/walkthrough sections; no live agent or coding-assistant session is required for this lab.

In this hands-on lab you'll work through 4 sections:

| # | Section | Outcome |
|---|---------|---------|
| 1 | Cortex AI Functions (Summarize, Sentiment, Translate) | run managed text AI from familiar SQL patterns |
| 2 | First AI Use Case Exercise on Customer Feedback Text | walk away with one applied pattern to run against a customer's own data |
| 3 | Snowflake CoWork | understand the natural-language agent layer over your own data |
| 4 | Snowflake CoCo — AI-Assisted Development Walkthrough | see how an AI coding assistant accelerates the exact work done in this lab |

### Setup

Create the lab database and tag this session.

In [ ]:
-- Attribution: tag this session's queries (no privilege needed; role-agnostic)
ALTER SESSION SET QUERY_TAG = '{"origin":"sf_ace","name":"ai-without-the-phd-cortex-ai","version":{"major":1,"minor":0},"attributes":{"program":"ace-webinar-series","webinar":"w05","source":"sql"}}';

USE ROLE SYSADMIN;
CREATE DATABASE IF NOT EXISTS AI_WITHOUT_THE_PHD_CORTEX_AI_HOL;
USE DATABASE AI_WITHOUT_THE_PHD_CORTEX_AI_HOL;
USE SCHEMA PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS AI_WITHOUT_THE_PHD_CORTEX_AI_WH
  WAREHOUSE_SIZE = XSMALL
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE
  COMMENT = 'Dedicated XS compute for this HOL; dropped in cleanup';
USE WAREHOUSE AI_WITHOUT_THE_PHD_CORTEX_AI_WH;

## Section 1 — Cortex AI Functions (Summarize, Sentiment, Translate)

**The problem:** Your team has thousands of rows of raw customer feedback but no time to read it and no ML infrastructure to process it at scale.


[Cortex AI functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql) bring managed AI inference into SQL. You call them with familiar SQL patterns, alongside functions such as `UPPER()` or `LENGTH()`, without building a separate model-serving layer for this lab.

This section covers three functions: **[SNOWFLAKE.CORTEX.SUMMARIZE](https://docs.snowflake.com/en/sql-reference/functions/summarize-snowflake-cortex)** is a scalar function that summarizes English text; **[SNOWFLAKE.CORTEX.SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex)** is a scalar function that returns a score from −1.0 to 1.0; and **[AI_TRANSLATE](https://docs.snowflake.com/en/sql-reference/functions/ai_translate)** translates text between supported languages. `AI_TRANSLATE` is the canonical surface for new translation use cases; the legacy `SNOWFLAKE.CORTEX.TRANSLATE` function is scheduled for deprecation by the end of 2026. Because these are scalar functions, you can compose them in a single `SELECT`, apply them across table rows, or chain them in a CTE — familiar SQL patterns.


> **Outcome:** run managed text AI from familiar SQL patterns.

### Confirm AI Function Access (do this first)

Cortex AI functions require **two** access layers. Confirm both before
the first `SUMMARIZE`, `SENTIMENT`, or `AI_TRANSLATE` call in this
section — missing either layer fails the lab mid-session.

1. the account-level `USE AI FUNCTIONS` privilege (or the matching
   per-function privileges), and
2. either the `SNOWFLAKE.CORTEX_USER` or
   `SNOWFLAKE.AI_FUNCTIONS_USER` database role.

Snowflake grants the blanket privilege and `CORTEX_USER` to `PUBLIC` by
default, but administrators can revoke either default. Confirm that the
active lab role already inherits both layers during account preflight.

> **Account-owner action only:** The next cell contains commented examples.
> It does not change account privileges. If access is missing, an
> `ACCOUNTADMIN` must approve and run the appropriate grant outside the lab.
> This package intentionally avoids leaving a persistent account-level grant.

> Docs: [Privileges and model access for Cortex AI Functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


In [ ]:
-- WALKTHROUGH ONLY — no statements in this cell execute.
-- Current access requires both an account privilege and a database role.
-- An ACCOUNTADMIN can approve and run grants such as:
-- USE ROLE ACCOUNTADMIN;
-- GRANT USE AI FUNCTIONS ON ACCOUNT TO ROLE SYSADMIN;
-- GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE SYSADMIN;
-- Source: https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access


### Step 1.1 — Set Up the Lab Context

This lab uses the SYSADMIN role to create the temporary database, warehouse, and tables, then points the session at the lab database and schema. In a production environment, use an approved custom role with only the required privileges.

**Why it matters:** Snowflake recommends avoiding ACCOUNTADMIN for routine object creation and using a role aligned to the workload.

In [ ]:
-- Step 1.1: Switch to the data-owning role and point the session at the lab DB
USE ROLE SYSADMIN;
USE DATABASE AI_WITHOUT_THE_PHD_CORTEX_AI_HOL;
USE SCHEMA PUBLIC;

### Step 1.2 — Create the Customer Feedback Table

Create `CUSTOMER_FEEDBACK_TABLE` — a realistic set of customer reviews that every AI function in this section and the next exercise will operate on.

Columns: `FEEDBACK_ID`, `CUSTOMER_NAME`, `FEEDBACK_TEXT` (the raw review), `SUBMITTED_DATE`, and `LANGUAGE` (a supported language code such as `en`, `fr`, or `es`). The non-English rows let us demo **AI_TRANSLATE** meaningfully.

**Why it matters:** These scalar functions accept string expressions, so a regular text column can be used directly in the function call. The same `SELECT` shape you use for `UPPER(feedback_text)` works for `SNOWFLAKE.CORTEX.SUMMARIZE(feedback_text)`.

> **Sample-data note:** The 15 reviews in this lab are synthetic teaching inputs. Product opinions, performance figures, and customer outcomes inside the quoted review text are fictional and are not Snowflake product claims or benchmarks.
>
> Docs: [Cortex AI Functions overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)

In [ ]:
-- Step 1.2: Create the feedback table (CREATE OR REPLACE is idempotent — safe to re-run)
CREATE OR REPLACE TABLE CUSTOMER_FEEDBACK_TABLE (
    feedback_id     INTEGER        NOT NULL,
    customer_name   VARCHAR(100),
    feedback_text   VARCHAR,                  -- raw review text; passed to every AI function
    submitted_date  DATE,
    language        VARCHAR(5)                -- supported language code: 'en' | 'fr' | 'es'
);

### Step 1.3 — Load Sample Feedback Rows

Insert 15 rows with deliberately varied content:

- **Length variety** — some rows are multi-sentence paragraphs (rows 1, 8, 12); others are short and punchy. Long rows make SUMMARIZE's compression visible; short rows make SENTIMENT's score more intuitive.
- **Sentiment variety** — rows 1, 2, 5, 7 are clearly positive; rows 3, 6, 8, 11 are clearly negative; rows 4, 13 are mixed.
- **Language variety** — rows 10, 14 are in French (`fr`); row 11 is in Spanish (`es`). These feed the AI_TRANSLATE demo.

**Why it matters:** real feedback data is messy — mixed lengths, mixed tones, mixed languages. This synthetic set exercises those shapes; exact AI output is model-generated and should be reviewed rather than treated as a benchmark.

In [ ]:
-- Step 1.3: Insert 15 sample rows — varied length, tone, and language for a rich demo
-- TRUNCATE first so re-running this cell alone stays idempotent (no duplicate rows)
TRUNCATE TABLE CUSTOMER_FEEDBACK_TABLE;
INSERT INTO CUSTOMER_FEEDBACK_TABLE VALUES
(1,  'Alex Johnson',
 'We migrated our entire data warehouse to Snowflake six months ago and the experience has been outstanding. Query performance improved dramatically — jobs that used to take 45 minutes now complete in under three. The auto-scaling feature means our BI team never waits during peak hours, and the separation of storage from compute has cut our monthly bill by 30 percent. The zero-copy cloning capability alone saved us weeks of work during our last audit. Highly recommend to any team still running on-prem data infrastructure.',
 '2024-10-15', 'en'),
(2,  'Emma Williams',
 'The support team resolved our critical production issue within two hours — on a Sunday. They stayed on the call the entire time, walked us through root cause analysis, and followed up Monday with a written summary and three recommendations to prevent recurrence. That level of support from a data platform vendor is rare.',
 '2024-11-02', 'en'),
(3,  'Michael Chen',
 'Billing is confusing and the credits system is not transparent. I got charged for compute I did not expect because a warehouse was left running over the weekend. There should be more aggressive auto-suspend defaults out of the box. Also the invoice breakdown does not distinguish between storage and compute clearly.',
 '2024-11-20', 'en'),
(4,  'Sarah Martinez',
 'Performance is generally good but we see occasional slowdowns during our monthly close process when multiple large queries hit the same warehouse simultaneously. We have tried increasing the warehouse size but the cost jumps significantly. Would like better visibility into which queries are causing the contention.',
 '2024-12-01', 'en'),
(5,  'David Lee',
 'The ML and AI features are exactly what our data science team needed. Cortex Search is surprisingly accurate out of the box and Cortex Analyst lets our business users ask questions in plain English without writing SQL. We went from a three-week model deployment cycle to same-day with the Model Registry.',
 '2025-01-10', 'en'),
(6,  'Jennifer Brown',
 'The UI is cluttered and inconsistent. Some features are buried under three or four menu levels and the navigation changed between versions without any in-app guidance. The SQL editor is good but worksheet management for large teams is painful — no folder hierarchy, no bulk tagging.',
 '2025-01-18', 'en'),
(7,  'Robert Davis',
 'Security controls are best-in-class. Dynamic data masking, row-level security, and column-level encryption all work together seamlessly. We passed our SOC 2 audit with zero findings related to our Snowflake environment. The network policy configuration was straightforward and the audit logs are comprehensive.',
 '2025-02-05', 'en'),
(8,  'Lisa Thompson',
 'We had a serious incident when a rogue query scanned several terabytes of historical data and consumed most of our monthly credit allocation in a single afternoon. There was no alerting until the damage was done. Resource monitors exist but the documentation on setting them up proactively is buried. This cost us a significant budget overrun and eroded trust with our finance team. We are still rebuilding confidence internally.',
 '2025-02-14', 'en'),
(9,  'James Wilson',
 'The REST API and Python connector are both excellent. Clean documentation, consistent behavior, and the Python connector handles retries and large result sets gracefully. Snowpark makes it easy to push transformation logic into Snowflake rather than pulling data out.',
 '2025-03-01', 'en'),
(10, 'Sophie Dubois',
 'La plateforme est tres intuitive et les performances sont impressionnantes. Notre equipe a reduit le temps de traitement de nos rapports hebdomadaires de deux heures a moins de dix minutes. Le support en francais serait un plus mais les documentations en anglais sont tres claires.',
 '2025-03-08', 'fr'),
(11, 'Carlos Rodriguez',
 'La integracion con nuestras herramientas existentes fue mas complicada de lo esperado. La documentacion sobre conectores de terceros esta desactualizada y tuvimos que abrir varios tickets de soporte antes de resolver los problemas de autenticacion. El producto es bueno pero la experiencia de configuracion inicial fue frustrante.',
 '2025-03-15', 'es'),
(12, 'Amanda Foster',
 'Onboarding was smooth and well-structured. The implementation team walked us through account setup, role hierarchy design, and resource monitors in a single session. Within two weeks we had our first production pipeline running. The Snowflake University courses were practical and directly applicable — not just theory.',
 '2025-03-22', 'en'),
(13, 'Kevin Park',
 'Some features work well, others feel incomplete. Time Travel is fantastic but the monitoring tooling around dynamic tables is still maturing. Overall we are cautiously optimistic but would not say we are fully satisfied yet.',
 '2025-04-01', 'en'),
(14, 'Marie Laurent',
 'Les fonctionnalites de gouvernance des donnees sont puissantes mais la courbe d''apprentissage est raide. Il a fallu plusieurs semaines pour comprendre comment structurer les roles et les politiques d''acces correctement. Une formation guidee sur ce sujet serait tres utile pour les nouvelles equipes.',
 '2025-04-10', 'fr'),
(15, 'Rachel Green',
 'The Snowflake partner ecosystem is a major differentiator. We connected Fivetran for ingestion, dbt for transformation, and Tableau for visualization in a single afternoon. Everything works together and the marketplace makes it easy to discover integrations.',
 '2025-04-15', 'en');

### Step 1.4 — Verify the Load and Preview the Data

Confirm all 15 rows loaded, then scan a few to see the variety in text length and language before running any AI on it.

**You should see:**
- `row_count = 15` from the COUNT query
- The `feedback_preview` column showing clearly different text lengths and tones
- Rows 10, 11, 14 with `language` values of `fr` or `es`

In [ ]:
-- Step 1.4a: Confirm row count
SELECT COUNT(*) AS row_count FROM CUSTOMER_FEEDBACK_TABLE;

In [ ]:
-- Step 1.4b: Preview — language column and first 100 chars of each review
SELECT
    feedback_id,
    customer_name,
    language,
    LEFT(feedback_text, 100) AS feedback_preview
FROM CUSTOMER_FEEDBACK_TABLE
ORDER BY feedback_id;

### Step 2.1 — Summarize Feedback Text with SNOWFLAKE.CORTEX.SUMMARIZE

`SNOWFLAKE.CORTEX.SUMMARIZE(text)` takes English text and returns a summary. Longer rows make it easier to compare the generated summary with the source.

> **Note:** SUMMARIZE processes English text. The query below filters to `language = 'en'` — use `AI_TRANSLATE` first (Step 4.1) to homogenize non-English rows before summarizing.

The query below gives the generated value an alias in a CTE so the outer projection can compare source and summary lengths without repeating the function expression in the SQL text.

**Why it matters:** analysts can use generated summaries as a triage aid, then inspect the raw text for rows that need attention. Validate the summaries on representative inputs before using them in a downstream workflow.

> Docs: [SNOWFLAKE.CORTEX.SUMMARIZE](https://docs.snowflake.com/en/sql-reference/functions/summarize-snowflake-cortex)

In [ ]:
-- Step 2.1: Summarize every English feedback row
-- Source: https://docs.snowflake.com/en/sql-reference/functions/summarize-snowflake-cortex
WITH summarized AS (
    SELECT
        feedback_id,
        customer_name,
        LENGTH(feedback_text)                            AS raw_char_count,
        SNOWFLAKE.CORTEX.SUMMARIZE(feedback_text)        AS ai_summary
    FROM CUSTOMER_FEEDBACK_TABLE
    WHERE language = 'en'
)
SELECT
    feedback_id,
    customer_name,
    raw_char_count,
    ai_summary,
    LENGTH(ai_summary)                                   AS summary_char_count
FROM summarized
ORDER BY raw_char_count DESC;  -- longest source text first for comparison

### Step 3.1 — Score Sentiment with SNOWFLAKE.CORTEX.SENTIMENT

`SNOWFLAKE.CORTEX.SENTIMENT(text)` returns a FLOAT between −1.0 and 1.0:
- **≥ 0.5** = positive
- **−0.5 to 0.5** = neutral
- **≤ −0.5** = negative

The CTE pattern below gives the score an alias and derives a readable label from the documented score bands.

**Why it matters:** a sentiment score turns unstructured text into an orderable, filterable number. Sort by `sentiment_score ASC` to surface the lowest-scoring feedback for review, or use a validated threshold in a follow-up workflow — all inside SQL with no external ML service.

**Inspect the result:** Compare the highest- and lowest-scoring rows with the source preview before adopting a routing rule. The documented score represents polarity and certainty, not sentiment intensity.

> Docs: [SNOWFLAKE.CORTEX.SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex)

In [ ]:
-- Step 3.1: Score sentiment on every English feedback row
-- Source: https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex
WITH scored AS (
    SELECT
        feedback_id,
        customer_name,
        feedback_text,
        SNOWFLAKE.CORTEX.SENTIMENT(feedback_text)        AS sentiment_score
    FROM CUSTOMER_FEEDBACK_TABLE
    WHERE language = 'en'
)
SELECT
    feedback_id,
    customer_name,
    ROUND(sentiment_score, 3)                            AS sentiment_score,
    CASE
        WHEN sentiment_score >=  0.5 THEN 'positive'
        WHEN sentiment_score <= -0.5 THEN 'negative'
        ELSE                              'neutral'
    END                                                  AS sentiment_label,
    LEFT(feedback_text, 100)                             AS feedback_preview
FROM scored
ORDER BY sentiment_score DESC;

### Step 3.2 — Sanity-Check: Do the Scores Match the Text?

Before trusting an AI score in a production workflow, verify it against human intuition on the extremes. This query surfaces the three highest and three lowest scores alongside the first 150 characters of each review.

**Why it matters:** Generated scores should be validated on representative inputs before they drive an alert or routing rule. If a score and its source text disagree, investigate the input and tune the business rule before production use.

**Inspect the result:** Review the highest- and lowest-scoring English-language rows alongside their source previews.

In [ ]:
-- Step 3.2: Top-3 highest and top-3 lowest English sentiment scores
WITH scored AS (
    SELECT
        feedback_id,
        customer_name,
        LEFT(feedback_text, 150)                         AS preview,
        ROUND(SNOWFLAKE.CORTEX.SENTIMENT(feedback_text), 3) AS score
    FROM CUSTOMER_FEEDBACK_TABLE
    WHERE language = 'en'
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (ORDER BY score DESC)  AS pos_rank,
        ROW_NUMBER() OVER (ORDER BY score ASC)   AS neg_rank
    FROM scored
)
SELECT
    CASE WHEN pos_rank <= 3 THEN 'highest score' ELSE 'lowest score' END  AS bucket,
    feedback_id,
    customer_name,
    score,
    preview
FROM ranked
WHERE pos_rank <= 3 OR neg_rank <= 3
ORDER BY score DESC;

### Step 4.1 — Translate Non-English Feedback with AI_TRANSLATE

`AI_TRANSLATE(text, source_language, target_language)` accepts text and two [supported language codes](https://docs.snowflake.com/en/sql-reference/functions/ai_translate), then returns translated text. Supported codes include `'en'`, `'fr'`, `'es'`, `'de'`, and `'ja'`; consult the docs for the current list. `AI_TRANSLATE` is the canonical surface for new use cases, while the legacy `SNOWFLAKE.CORTEX.TRANSLATE` function is scheduled for deprecation by the end of 2026.

Rows 10 and 14 are in French (`fr`); row 11 is in Spanish (`es`). The query passes the `language` column directly as the source-language argument — a clean pattern when your table already carries a language tag.

**Why it matters:** supported-language inputs can be normalized to English before the English-only SUMMARIZE and SENTIMENT steps, keeping one downstream SQL pattern.

**You should see:**
- Rows 10 and 14: a generated English translation of the French text
- Row 11: a generated English translation of the Spanish text
- Source and translated text available side by side for review

> Docs: [AI_TRANSLATE](https://docs.snowflake.com/en/sql-reference/functions/ai_translate)

In [ ]:
-- Step 4.1: Translate non-English rows to English using AI_TRANSLATE (replaces deprecated SNOWFLAKE.CORTEX.TRANSLATE)
-- The `language` column holds a supported source-language code; pass it directly as the second argument
-- Source: https://docs.snowflake.com/en/sql-reference/functions/ai_translate
SELECT
    feedback_id,
    customer_name,
    language                                                           AS source_language,
    feedback_text                                                      AS original_text,
    AI_TRANSLATE(feedback_text, language, 'en')                        AS translated_to_english
FROM CUSTOMER_FEEDBACK_TABLE
WHERE language <> 'en'
ORDER BY feedback_id;

### Step 5.1 — Compose SUMMARIZE + SENTIMENT in One Query

Because these are scalar SQL functions, you can compose them in a single SELECT with no intermediate tables. This cell runs SUMMARIZE and SENTIMENT together across the English rows and returns one result set for review.

This composability pattern is the foundation of the next section's exercise: a full pipeline that enriches all rows (including translated ones) with summary, sentiment score, and label in one query.

**Why it matters:** a single query returning `(feedback_id, ai_summary, sentiment_score, sentiment_label)` produces a useful review shape. After validating the generated values and business rules, you can store the result for a dashboard or use it in a follow-up workflow. All in SQL, without training a custom model or operating a separate model-serving layer for this lab.

In [ ]:
-- Step 5.1: Compose SUMMARIZE + SENTIMENT together — lowest scores first for review
-- Source (summarize): https://docs.snowflake.com/en/sql-reference/functions/summarize-snowflake-cortex
-- Source (sentiment): https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex
WITH enriched AS (
    SELECT
        feedback_id,
        customer_name,
        SNOWFLAKE.CORTEX.SUMMARIZE(feedback_text)              AS ai_summary,
        SNOWFLAKE.CORTEX.SENTIMENT(feedback_text)              AS sentiment_score
    FROM CUSTOMER_FEEDBACK_TABLE
    WHERE language = 'en'
)
SELECT
    feedback_id,
    customer_name,
    ai_summary,
    ROUND(sentiment_score, 3)                                  AS sentiment_score,
    CASE
        WHEN sentiment_score >=  0.5 THEN 'positive'
        WHEN sentiment_score <= -0.5 THEN 'negative'
        ELSE                              'neutral'
    END                                                        AS sentiment_label
FROM enriched
ORDER BY sentiment_score ASC;  -- lowest scores first for review

### Verify the Feedback Table in Snowsight (read-only)

[Open in Snowsight](https://app.snowflake.com/_deeplink/#/data/databases/AI_WITHOUT_THE_PHD_CORTEX_AI_HOL)

Read-only — confirm what you just built; all objects were created by the SQL above.

> **Apply on your account**
>
> **Apply on your account:**
> - Replace `CUSTOMER_FEEDBACK_TABLE` with your real feedback, review, ticket, or support-case table.
> - These functions accept string expressions — swap `feedback_text` for the text expression you need to process.
> - `AI_TRANSLATE` accepts documented language codes; store a supported code in a `language` column and pass it as the source-language argument. To auto-detect a supported source language, pass an empty string `''` as the second argument.
> - The SENTIMENT threshold (±0.5) is the documented boundary — run the score distribution on your real data first and tune if you need a stricter or looser definition of "actionable negative."
> - These are managed SQL functions; permissions and model availability still follow your account configuration, region, and active role.
> - Access control: confirm the two layers from the access preflight at the start of this section — `USE AI FUNCTIONS` (or matching per-function privileges) plus `SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.AI_FUNCTIONS_USER` — before the first function call. The blanket privilege and `CORTEX_USER` are granted to `PUBLIC` by default, but administrators can revoke those defaults.


## Section 2 — First AI Use Case Exercise on Customer Feedback Text

**The problem:** Support and account teams drown in unread feedback rows they can't prioritize fast
enough — reading every ticket, review, or survey response manually doesn't scale,
and the customers who need urgent attention get lost in the noise.


This exercise wires the Cortex AI functions from the previous section into a single
SQL pipeline. You take **CUSTOMER_FEEDBACK_TABLE**, normalize supported non-English
rows with **[AI_TRANSLATE](https://docs.snowflake.com/en/sql-reference/functions/ai_translate)**,
run
**[SNOWFLAKE.CORTEX.SUMMARIZE](https://docs.snowflake.com/en/sql-reference/functions/summarize-snowflake-cortex)**
and
**[SNOWFLAKE.CORTEX.SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex)**
in an `INSERT … SELECT`, derive a business rule (`needs_followup = TRUE`
when the score is below zero), and land the result in **FEEDBACK_INSIGHTS_TABLE** — a
structured review table your team can query, dashboard, or use in a validated follow-up workflow.


> **Outcome:** walk away with one applied pattern to run against a customer's own data.

## First AI Use Case Exercise — Automated Feedback Triage

You've seen each Cortex AI function work individually. Now we wire them together
into a **reusable SQL pipeline**: ingest raw customer feedback, normalize the
supported French and Spanish inputs, run AI enrichment, and surface rows that
meet a review rule — all without leaving SQL.

**Why it matters:** This is a concrete starting pattern for support tickets,
survey responses, or review feeds. Validate the generated output and tune the
review rule on representative data before production use.

> Docs: [Snowflake Cortex AI Functions — Overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)

Access for the Cortex AI function calls in this exercise was confirmed
at the start of Section 1 (`USE AI FUNCTIONS` plus `CORTEX_USER` or
`AI_FUNCTIONS_USER`).


In [ ]:
-- Switch to the data-owning role and set session context
USE ROLE SYSADMIN;
USE DATABASE AI_WITHOUT_THE_PHD_CORTEX_AI_HOL;
USE SCHEMA PUBLIC;

### Step 1 — Review the Source Data

Preview **CUSTOMER_FEEDBACK_TABLE** — the input to our pipeline. This table
was created in the previous section and holds raw, unstructured `feedback_text`
alongside `feedback_id`, `customer_name`, and `submitted_date`.

**Why it matters:** Understanding the input shape — length, language, messiness
of the text — sets expectations for what the AI output will look like.

**You should see:** Rows of raw feedback text, full-length, unanalyzed.

> Docs: [SNOWFLAKE.CORTEX.SUMMARIZE](https://docs.snowflake.com/en/sql-reference/functions/summarize-snowflake-cortex)


In [ ]:
-- Step 1: Preview source rows before enrichment
SELECT
    feedback_id,
    customer_name,
    LEFT(feedback_text, 120) AS feedback_preview,   -- truncated for display
    submitted_date,
    language
FROM CUSTOMER_FEEDBACK_TABLE
LIMIT 10;


### Step 2 — Create the Insights Table

Create **FEEDBACK_INSIGHTS_TABLE** — the structured output of the AI pipeline.
It stores one enriched row per feedback item: a generated `summary`, a numeric
`sentiment_score` (−1.0 to 1.0, representing polarity and certainty), a categorical
`sentiment_label`, a `needs_followup` boolean flag, and a `processed_at`
timestamp.

The thresholds for `sentiment_label` follow the
[SENTIMENT function's documented scale](https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex):
POSITIVE ≥ 0.5, NEGATIVE ≤ −0.5, NEUTRAL in between.

**Why it matters:** Keeping AI output in a dedicated table leaves the raw
source intact, gives downstream tools a clean typed surface, and makes the
pipeline safely re-runnable — `CREATE OR REPLACE` re-creates the table clean.

**Run as:** SYSADMIN — creates and owns data objects.

> Docs: [SNOWFLAKE.CORTEX.SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex)
> · [SNOWFLAKE.CORTEX.SUMMARIZE](https://docs.snowflake.com/en/sql-reference/functions/summarize-snowflake-cortex)


In [ ]:
-- Step 2: Create (or replace) the insights table — idempotent on re-run
-- Source: https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex
USE ROLE SYSADMIN;
CREATE OR REPLACE TABLE FEEDBACK_INSIGHTS_TABLE (
    feedback_id     NUMBER,
    customer_name   VARCHAR(256),
    summary         TEXT,               -- AI-generated plain-English summary
    sentiment_score FLOAT,              -- -1.0 to 1.0 polarity/certainty score
    sentiment_label VARCHAR(16),        -- 'POSITIVE' | 'NEUTRAL' | 'NEGATIVE' (per docs thresholds)
    needs_followup  BOOLEAN,            -- TRUE when sentiment_score < 0
    processed_at    TIMESTAMP_NTZ       -- when this row was enriched
);


### Step 3 — Run the Combined AI Enrichment Pipeline

The core of the exercise is one `INSERT … SELECT`. A normalization CTE keeps
English text as-is and translates the supported French and Spanish rows to
English before **SUMMARIZE** and **SENTIMENT** process them.

The next CTE gives the generated summary and sentiment score stable aliases.
The final projection derives the label and `needs_followup` flag from the score.

The business rule: `needs_followup = sentiment_score < 0` flags any row with
a negative lean. This review rule is more inclusive than the docs' NEGATIVE
band, which begins at −0.5. Validate the rule against your own
labeled examples before automating an action.

**Why it matters:** Replace `CUSTOMER_FEEDBACK_TABLE` with a customer's
support-ticket table and this pattern becomes a reusable starting point. The
enrichment logic stays in SQL through managed Cortex AI functions.

> Docs: [SNOWFLAKE.CORTEX.SUMMARIZE](https://docs.snowflake.com/en/sql-reference/functions/summarize-snowflake-cortex)
> · [SNOWFLAKE.CORTEX.SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex)
> · [AI_TRANSLATE](https://docs.snowflake.com/en/sql-reference/functions/ai_translate)


In [ ]:
-- Step 3: Normalize supported languages, then enrich every feedback row
-- Translation source: https://docs.snowflake.com/en/sql-reference/functions/ai_translate
-- Source: https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex
INSERT INTO FEEDBACK_INSIGHTS_TABLE
WITH normalized AS (
    SELECT
        feedback_id,
        customer_name,
        CASE
            WHEN language = 'en' THEN feedback_text
            ELSE AI_TRANSLATE(feedback_text, language, 'en')
        END AS feedback_en
    FROM CUSTOMER_FEEDBACK_TABLE
),
base AS (
    SELECT
        feedback_id,
        customer_name,
        SNOWFLAKE.CORTEX.SUMMARIZE(feedback_en)   AS summary,
        SNOWFLAKE.CORTEX.SENTIMENT(feedback_en)   AS sentiment_score
    FROM normalized
)
SELECT
    feedback_id,
    customer_name,
    summary,
    sentiment_score,
    CASE
        WHEN sentiment_score >=  0.5 THEN 'POSITIVE'   -- docs: 0.5 to 1
        WHEN sentiment_score <= -0.5 THEN 'NEGATIVE'   -- docs: -0.5 to -1
        ELSE                              'NEUTRAL'     -- docs: -0.5 to 0.5
    END                        AS sentiment_label,
    sentiment_score < 0        AS needs_followup,   -- TRUE = any negative lean = flag for review
    CURRENT_TIMESTAMP()        AS processed_at
FROM base;


### Step 4 — Preview the Enriched Results

Compare the insights table against the raw source to confirm the AI output
looks correct.

**You should see:**
- An AI-generated `summary` column for each processed row
- A `sentiment_score` float between −1.0 and 1.0
- A `sentiment_label` of POSITIVE, NEUTRAL, or NEGATIVE
- A `needs_followup` boolean for every row

> Docs: [SNOWFLAKE.CORTEX.SENTIMENT — return value and scale](https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex)


In [ ]:
-- Step 4: Preview the enriched insights table
SELECT
    feedback_id,
    customer_name,
    sentiment_label,
    needs_followup,
    ROUND(sentiment_score, 3)   AS score,
    LEFT(summary, 200)          AS summary_preview
FROM FEEDBACK_INSIGHTS_TABLE
LIMIT 10;


### Step 5 — Surface the Feedback That Needs Attention

This is the business payoff: one `WHERE` clause turns the insights table into
a review queue. Ordering by `sentiment_score ASC` puts the lowest-scoring
feedback first.

**Why it matters:** A support lead or account manager can now review the rows
that meet the chosen follow-up rule in a single query. The score is a model
output to inspect, not a measure of customer urgency.

**You should see:** Only rows whose score is below the teaching rule's zero
threshold, ordered from lowest score upward. Some rows can still carry a
NEUTRAL label because the documented NEGATIVE band starts at −0.5.

> Docs: [SNOWFLAKE.CORTEX.SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex)


In [ ]:
-- Step 5: Review rows that meet the chosen follow-up rule
SELECT
    feedback_id,
    customer_name,
    sentiment_label,
    ROUND(sentiment_score, 3)   AS score,
    summary
FROM FEEDBACK_INSIGHTS_TABLE
WHERE needs_followup = TRUE
ORDER BY sentiment_score ASC;   -- lowest score first


### Step 6 — The Other Side: Feedback That's Fine

Equally important — review which rows did not meet the follow-up rule.

**Why it matters:** A triage system is only trustworthy when both outputs are
reviewable. This query lets a manager spot-check unflagged rows before relying
on the review rule.

**You should see:** Rows with POSITIVE or NEUTRAL labels and `needs_followup =
FALSE`, ordered from the highest score downward.

> Docs: [SNOWFLAKE.CORTEX.SENTIMENT — sentiment score guidance](https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex)


In [ ]:
-- Step 6: Rows that did not meet the follow-up rule
SELECT
    feedback_id,
    customer_name,
    sentiment_label,
    ROUND(sentiment_score, 3)   AS score,
    summary
FROM FEEDBACK_INSIGHTS_TABLE
WHERE needs_followup = FALSE
ORDER BY sentiment_score DESC;  -- highest score first


### Step 7 — Inspect the Multilingual Normalization

The core pipeline already uses
**[AI_TRANSLATE](https://docs.snowflake.com/en/sql-reference/functions/ai_translate)**
as its normalization stage. `AI_TRANSLATE` is the current surface for new use
cases; the legacy `SNOWFLAKE.CORTEX.TRANSLATE` function is scheduled for
deprecation by the end of 2026.

This optional read-only query isolates the French and Spanish rows so you can
compare each original with the English text passed downstream. It uses the
stored source-language code; passing `''` instead would ask AI_TRANSLATE to
auto-detect a supported source language.

**Why it matters:** Inspecting normalized text is a practical quality gate
before summaries and scores drive downstream actions.

> Docs: [AI_TRANSLATE](https://docs.snowflake.com/en/sql-reference/functions/ai_translate)
> · [SNOWFLAKE.CORTEX.SUMMARIZE](https://docs.snowflake.com/en/sql-reference/functions/summarize-snowflake-cortex)


In [ ]:
-- Step 7 (optional): Inspect the non-English rows and their normalized text
-- Source: https://docs.snowflake.com/en/sql-reference/functions/ai_translate
SELECT
    feedback_id,
    customer_name,
    language AS source_language,
    feedback_text AS original_text,
    AI_TRANSLATE(feedback_text, language, 'en') AS normalized_english
FROM CUSTOMER_FEEDBACK_TABLE
WHERE language <> 'en'
ORDER BY feedback_id;


### Step 8 — Chart the Sentiment Distribution

Roll the enriched labels into one row per `sentiment_label`. This is
the visual payoff of the pipeline — the mix of POSITIVE, NEUTRAL, and
NEGATIVE feedback — before the CoWork section shows how a business user
can ask for the same view in natural language.

The SQL cell below shows the distribution as an interactive table; the Python
cell right after it renders the same distribution as a bar chart inline — no
manual chart setup.

**Why it matters:** A distribution makes the enrichment tangible. Exact
counts vary because `SNOWFLAKE.CORTEX.SENTIMENT` is model-generated;
treat the bars as output to inspect, not as a benchmark.

**You should see:** A bar chart with one bar per sentiment label present
in `FEEDBACK_INSIGHTS_TABLE`, ordered NEGATIVE, NEUTRAL, POSITIVE, with
counts that sum to 15.

> Docs: [SNOWFLAKE.CORTEX.SENTIMENT — return value and scale](https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex)


In [ ]:
-- Step 8: Sentiment distribution — aggregated counts per label
-- The Python cell below renders the same distribution as a bar chart
-- Source: https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex
SELECT
    sentiment_label,
    COUNT(*)                              AS feedback_count,
    ROUND(AVG(sentiment_score), 3)        AS avg_score,
    SUM(IFF(needs_followup, 1, 0))        AS followup_rows
FROM FEEDBACK_INSIGHTS_TABLE
GROUP BY sentiment_label
ORDER BY
    CASE sentiment_label
        WHEN 'NEGATIVE' THEN 1
        WHEN 'NEUTRAL'  THEN 2
        WHEN 'POSITIVE' THEN 3
        ELSE 4
    END;


In [ ]:
# Step 8 (chart): render the sentiment distribution inline.
# Self-contained: query via the active Snowpark session so the chart never depends on a
# cross-cell result pointer (dataframe_x), which is position-dependent and differs by surface/runtime.
# Ref: https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-in-workspaces/notebooks-in-workspaces-edit-run
import os
os.environ["MPLCONFIGDIR"] = "/tmp/"   # matplotlib cache dir must be writable in notebooks
import matplotlib.pyplot as plt
from snowflake.snowpark.context import get_active_session

session = get_active_session()
dist = session.sql("""
    SELECT sentiment_label, COUNT(*) AS feedback_count
    FROM AI_WITHOUT_THE_PHD_CORTEX_AI_HOL.PUBLIC.FEEDBACK_INSIGHTS_TABLE
    GROUP BY sentiment_label
    ORDER BY CASE sentiment_label
                 WHEN 'NEGATIVE' THEN 1 WHEN 'NEUTRAL' THEN 2 WHEN 'POSITIVE' THEN 3 ELSE 4 END
""").to_pandas()

colors = {'NEGATIVE': '#e74c3c', 'NEUTRAL': '#f39c12', 'POSITIVE': '#2ecc71'}
bar_colors = [colors.get(l, '#3498db') for l in dist['SENTIMENT_LABEL']]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(dist['SENTIMENT_LABEL'], dist['FEEDBACK_COUNT'], color=bar_colors)
ax.set_xlabel('Sentiment label'); ax.set_ylabel('Feedback count')
ax.set_title('Customer feedback by sentiment label')
plt.tight_layout(); plt.show()

### Apply This Pattern on Your Own Data

You just built a reusable AI triage pipeline. To run it against a customer's
real data, swap the source table and adjust column names:

- **Support tickets** — replace `CUSTOMER_FEEDBACK_TABLE` with the ticket table;
  `feedback_text` = ticket body; add `ticket_id`, `priority`, `assignee` columns
- **Product reviews** — same pattern; add a `product_id` column to
  FEEDBACK_INSIGHTS_TABLE and group by product downstream
- **Survey responses** — one INSERT per question column; union the results
  for a cross-question sentiment dashboard

The `needs_followup` flag plugs directly into other Snowflake features:
- A **Snowflake Alert** to notify the team when new flagged rows arrive
- A BI dashboard `WHERE needs_followup = TRUE` filter for a live triage view
- A downstream CRM write to auto-create a follow-up task per flagged row

**The three-line version for your next customer conversation:** "We took your
raw feedback table, ran three managed SQL functions, and got a review queue based
on a rule we can validate. The pattern stays close to the data and can be adapted
to your workflow."

## Section 3 — Snowflake CoWork

> **Walkthrough only:** Snowflake CoWork is accessed through Snowsight and a dedicated conversational interface. There is no SQL to run in this section — walk through the concepts and entry points here, then explore it in your own account.

**The Problem:** The pipeline in Section 2 built `FEEDBACK_INSIGHTS_TABLE` with sentiment labels, scores, summaries, and follow-up flags. But answering questions like *"which feedback rows have the lowest sentiment this month?"* usually requires writing SQL, opening a ticket, or building a custom dashboard.

**The Solution:** [Snowflake CoWork](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-cowork) (GA November 4, 2025; formerly "Snowflake Intelligence") is an out-of-the-box agentic application that enables non-technical users to query enterprise data conversationally.

**How It Works:**
1. **User asks a question** in plain English (e.g., *"Show negative feedback trends this month"*).
2. **Cortex Agent orchestrates:** Evaluates intent and selects the right tools (semantic views, Cortex Search, or custom tools).
3. **Cortex Analyst generates SQL:** Translates the natural-language question into governed SQL executed against a [semantic view](https://docs.snowflake.com/en/user-guide/views-semantic/overview), respecting all role privileges and row/column policies.
4. **Governed response delivered:** Returns the answer in natural language accompanied by an interactive chart or table with full query traceability.

> **Outcome:** Authorized stakeholders can explore enriched data independently without writing SQL or waiting on data teams.
>
> **Docs:** [Overview of Snowflake CoWork](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-cowork) · [GA Release Notes (Nov 4, 2025)](https://docs.snowflake.com/en/release-notes/2025/other/2025-11-04-snowflake-intelligence)

### Step 3.1 — How to Access CoWork


To access Snowflake CoWork in Snowsight:

1. Sign in to Snowsight
2. In the left navigation menu, go to **AI & ML → Snowflake CoWork**
3. Select an agent (each agent is configured with a specific data context — the semantic views and search services it knows about) and type your question in the chat panel

Administrators can also manage, create, and configure underlying agents under **AI & ML → Agents**.

**Why it matters:** CoWork automatically inherits and respects existing Snowflake governance controls, including row-access policies and column-level security. Users still need `USAGE` on the agent, database, and schema plus the privileges required by each attached tool and underlying object, so access remains an explicit deployment step.

> Docs: [User access and settings for agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-cowork/deploy-agents)

### Step 3.2 — How CoWork Gets Grounded in Your Data

For structured-data questions, Cortex Analyst uses a **semantic view**: column names, data types, table relationships, business metric definitions, and descriptions that connect business language to the underlying schema. This grounding helps the agent generate SQL against consistent business definitions.

**The architecture (from the CoWork docs):**

| Layer | Role |
|---|---|
| **Snowflake CoWork** | Conversational UI; routes questions to the right agent |
| **Cortex Agent** | Orchestrates the conversation; selects tools; plans the action sequence |
| **Cortex Analyst** | Translates natural language to SQL using the semantic view definition |
| **Semantic View** | The business-meaning layer: column descriptions, metric formulas, join paths, business terms |
| **Physical Table** | The actual data — e.g. `FEEDBACK_INSIGHTS_TABLE` |

Per the docs: "Semantic views address the mismatch between how business users describe data and how it's stored in database schemas. With semantic views, you can define business metrics and model business entities and their relationships."

A semantic view is the recommended grounding layer for Cortex Analyst's structured-data tool. Other agent tools can address other data and action patterns, but the semantic view is the bridge used in this example between "what the user asks" and "what SQL to run."

**Connecting this to what we just built:**

The `FEEDBACK_INSIGHTS_TABLE` created in the previous section has columns: `sentiment_score` (float, −1.0 to +1.0), `sentiment_label` (POSITIVE / NEUTRAL / NEGATIVE), `needs_followup` (boolean), `summary` (AI-generated plain-English summary), and `processed_at` (enrichment timestamp). A semantic view on top of that table would define:

- `sentiment_score` — column description: "Polarity and certainty score from −1.0 to +1.0, generated by `SNOWFLAKE.CORTEX.SENTIMENT`; it does not measure sentiment intensity"
- `sentiment_label` — dimension: `POSITIVE`, `NEUTRAL`, or `NEGATIVE`, derived at the ±0.5 threshold
- `needs_followup` — dimension: boolean flag set to `TRUE` when `sentiment_score < 0`
- `avg_sentiment` — metric: `AVG(sentiment_score)`, sliceable by label or time period
- `followup_rate` — metric: `SUM(CASE WHEN needs_followup THEN 1 ELSE 0 END) / COUNT(*)`

With that semantic view in place and connected to a CoWork agent, an authorized business user can ask: *"How many feedback rows need follow-up this month, and what is the average sentiment score for negative feedback?"*

**Why it matters:** You don't need to rebuild the enrichment pipeline for each question. The Cortex AI functions produce a reusable structured table; a semantic view and correctly permissioned agent make that result available for conversational exploration.

> Docs: [Overview of semantic views](https://docs.snowflake.com/en/user-guide/views-semantic/overview) · [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)

### Step 3.3 — Example: Asking CoWork About Enriched Feedback

Here's what a business user's CoWork session looks like against the data built in this lab — after a semantic view has been created on `FEEDBACK_INSIGHTS_TABLE`.

**User types in CoWork:**
> "What's the sentiment breakdown in customer feedback this month, and how many customers need follow-up?"

**CoWork (via Cortex Analyst on the semantic view) generates and runs:**

```sql
/* WALKTHROUGH ONLY — Cortex Analyst generates this internally; the user never writes SQL */
SELECT
    sentiment_label,
    COUNT(*)                                              AS feedback_count,
    ROUND(AVG(sentiment_score), 2)                       AS avg_score,
    SUM(CASE WHEN needs_followup THEN 1 ELSE 0 END)     AS followup_count
FROM FEEDBACK_INSIGHTS_TABLE
WHERE DATE_TRUNC('month', processed_at) = DATE_TRUNC('month', CURRENT_DATE())
GROUP BY sentiment_label
ORDER BY avg_score ASC;
/* END WALKTHROUGH */
```

**CoWork can return a natural-language summary with a chart or table.** The exact counts and averages depend on the current table contents and generated sentiment scores, so this walkthrough intentionally does not hard-code an answer.

**What just happened — tracing the full arc:**

- `SNOWFLAKE.CORTEX.SENTIMENT` in Section 1 scored raw feedback text; the Section 2 pipeline stored results as structured columns in `FEEDBACK_INSIGHTS_TABLE`
- A semantic view defined what those columns *mean* in business language — `sentiment_label` is a dimension, `avg_score` is a metric, `processed_at` is the time axis
- The user asked a business question in plain English
- **Cortex Analyst** translated the intent to SQL using the semantic view, ran it, and CoWork packaged the result as a chart + natural-language summary
- The end user did not need to write the generated SQL

This is the payoff of the enrichment work done in the prior sections. The Cortex AI functions turned unstructured text into structured insights. CoWork gives authorized users a conversational path to those insights.

> Docs: [Overview of Snowflake CoWork](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-cowork)

### Step 3.4 — CoWork vs Cortex AI Functions: When to Use Each

**Cortex AI functions** and **Snowflake CoWork** solve different problems. They're complementary — not competing.

| | Cortex AI Functions | Snowflake CoWork |
|---|---|---|
| **Who uses it** | Data engineers, analysts writing SQL | Business users, execs, analysts |
| **How you interact** | SQL function call in a query or pipeline | Natural-language chat |
| **Best for** | Repeatable pipelines, bulk enrichment, data processing | Ad-hoc exploration, one-off questions, self-serve analysis |
| **Example** | `SNOWFLAKE.CORTEX.SENTIMENT(feedback_text)` on every row | "Which feedback rows had the lowest sentiment scores last month?" |
| **Output** | A column in a table you own | Conversational answer + chart or table |
| **Scales to** | Production workloads, scheduled tasks | Interactive sessions |

**The combined pattern — enrich once, explore repeatedly:**

Use `SNOWFLAKE.CORTEX.SUMMARIZE` and `SNOWFLAKE.CORTEX.SENTIMENT` (the scalar, per-row functions used in Sections 1 and 2) to enrich raw feedback text and store the results in `FEEDBACK_INSIGHTS_TABLE`. That can be a one-time or scheduled operation. Then create and evaluate a semantic view on top of `FEEDBACK_INSIGHTS_TABLE`, connect it to a CoWork agent, and grant authorized users the required privileges. Those users can then explore the enriched data conversationally.

**The Cortex AI functions perform repeatable enrichment. CoWork provides a conversational path to the governed result.**

> Docs: [Cortex AI Functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql) · [Snowflake CoWork](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-cowork)

### Step 3.5 — Enabling CoWork on Your Account

Snowflake CoWork is generally available. This lab keeps setup conceptual because creating and deploying an agent requires account-specific data, privileges, and tool configuration. The three high-level steps are:

**Step 1 — Create an agent (admin, Snowsight):**

An agent is the object that defines what data context CoWork reasons about. In Snowsight: go to **AI & ML → Agents → + Agent**. Connect the agent to a semantic view on your enriched table. The [Getting Started guide](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-cowork/getting-started) walks through creating your first enterprise agent with both structured and unstructured data.

**Step 2 — Control user access:**

Per the docs: "Administrators can use existing identity providers to give teams access only to Snowflake CoWork, making sure users only interact with the data experiences built for them." Access uses Snowflake's existing role system — users only see data their active role can already access. Refer to the [Getting Started guide](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-cowork/getting-started) for the current privilege model, as access controls for CoWork follow the Cortex Agents privilege structure.

**Step 3 — Build the semantic view (the real investment):**

Create a semantic view on `FEEDBACK_INSIGHTS_TABLE`. Add:
- Column descriptions explaining what each column means in business language (e.g., "`sentiment_score`: float from −1.0 to +1.0 generated by AI sentiment analysis")
- Metric definitions for common aggregations (`avg_sentiment`, `followup_rate`)
- Dimension descriptions for categorical columns like `sentiment_label`

Per the docs: "Semantic views address the mismatch between how business users describe data and how it's stored in database schemas." The semantic view supplies reusable definitions for **Cortex Analyst**, BI integrations, and other applications that use the data model.

**Why it matters:** A well-defined semantic view captures reusable business meaning. Agent configuration, evaluation, and privileges remain part of deploying the experience safely as questions evolve.

> Docs: [Getting started with Snowflake CoWork](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-cowork/getting-started) · [Overview of semantic views](https://docs.snowflake.com/en/user-guide/views-semantic/overview)

> **Apply on your account**
>
> On your account: model the tables enriched by your Cortex AI function pipelines in a semantic view. Add column descriptions and metric definitions for your highest-traffic business questions, then evaluate representative questions and generated SQL. The `FEEDBACK_INSIGHTS_TABLE` — with its `sentiment_label`, `sentiment_score`, `needs_followup`, and `summary` columns — is a useful starting shape. A deployed experience also requires an agent, the relevant tools, and explicit privileges for its authorized users.

## Section 4 — Snowflake CoCo — AI-Assisted Development Walkthrough

> **Walkthrough only:** This section has no SQL cells to run. The prompts below can be run directly in CoCo in Snowsight, CoCo Desktop, or the CoCo CLI (`cortex`).

**The Problem:** Iterating on data pipelines often introduces friction: looking up exact function syntax, parsing complex error traces, or finding relevant documentation instead of focusing on business logic.

**The Solution:** **[Snowflake CoCo](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code)** (formerly Cortex Code) is Snowflake's AI-driven developer assistant. It generates, explains, refactors, and debugs SQL and Python using the schema and environment context available in your Snowflake account.

> **Outcome:** See how an AI coding assistant accelerates the exact tasks performed across this lab.

### Snowflake CoCo — Available Surfaces

| Surface | Description | Status | Best for |
|:---|:---|:---:|:---|
| **Snowsight** | Native assistant in the Snowflake web UI | GA | Inline SQL fixes, schema queries, quick explanations |
| **[CoCo Desktop](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code-desktop)** | AI-powered desktop IDE (macOS & Windows) | GA | Local repos, multi-file projects, dbt, Streamlit apps |
| **[CoCo CLI](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code-cli)** | Terminal agent (`cortex`) | GA | Shell scripting, automation, local development |

*Note for live presenters:* If presenting from **CoCo Desktop**, the IDE you are currently using is the product being demonstrated. You can open the chat panel and run any of the prompts below live.

> **Docs:** [CoCo Overview](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code) · [CoCo Desktop](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code-desktop) · [CoCo CLI](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code-cli) · [Snowsight GA Release](https://docs.snowflake.com/en/release-notes/2026/other/2026-03-09-cortex-code-snowsight-ga)

### Three Prompts to Try — From This Lab's Content

Copy, paste, and adapt these prompts directly into the CoCo chat panel in Snowsight or CoCo Desktop:

---

**Prompt 1 — Verify an AI Function Edge Case**
```
Using current Snowflake documentation, explain what SNOWFLAKE.CORTEX.SENTIMENT returns when the input text is NULL. If the reference page does not specify that edge case, propose a minimal read-only query to test it in this account.
```
*What CoCo does:* Searches current Snowflake documentation, identifies whether the `NULL` edge case is explicitly documented, and generates a minimal probe query (e.g., `SELECT SNOWFLAKE.CORTEX.SENTIMENT(NULL);`).

---

**Prompt 2 — Debug an Invalid Identifier Error**
```
Fix this Snowflake SQL error:
Error: invalid identifier 'FEEDBACK_BODY'.

My SQL:
SELECT
  feedback_id,
  SNOWFLAKE.CORTEX.SENTIMENT(feedback_body) AS sentiment_score
FROM CUSTOMER_FEEDBACK_TABLE;
```
*What CoCo does:* Inspects schema context from `CUSTOMER_FEEDBACK_TABLE`, identifies that the correct column is `feedback_text`, and returns the corrected query.

---

**Prompt 3 — Generate a Streamlit Visualization**
```
Build a Streamlit app that shows a bar chart of feedback counts by sentiment_label (POSITIVE, NEGATIVE, NEUTRAL) from FEEDBACK_INSIGHTS_TABLE in the AI_WITHOUT_THE_PHD_CORTEX_AI_HOL database.
```
*What CoCo does:* Generates clean Python code using `st.connection("snowflake")` or Snowpark session to aggregate `sentiment_label` and display an interactive chart.

---

> **Tip:** Always inspect generated SQL and schema references prior to executing queries against your account.
>
> **Docs:** [SNOWFLAKE.CORTEX.SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex) · [Streamlit in Snowflake](https://docs.snowflake.com/en/developer-guide/streamlit/about-streamlit)

### Four Entry Points to AI Without the PhD

Here is the full arc of this lab — four entry points at different
levels of abstraction:

| Tool | What it does | How you reach it |
|------|-------------|-----------------|
| **Cortex AI functions** | Run managed text AI as SQL function calls without training a custom model | `SNOWFLAKE.CORTEX.SENTIMENT()`, `SNOWFLAKE.CORTEX.SUMMARIZE()`, `AI_TRANSLATE()` in a SQL query |
| **First AI Use Case exercise** | Apply those functions to a feedback-triage question | SQL + an existing text table |
| **Snowflake CoWork** | Ask your data questions in plain English — natural-language agent over your Snowflake account | Natural language in the CoWork interface |
| **Snowflake CoCo** | Generate, fix, and iterate on SQL and code from a description or an error message | Natural language in the CoCo panel |

In this lab, the managed functions run without training a custom model
or operating a separate model-serving layer.

**The acceleration compounds:** Cortex AI functions make your text
data AI-ready. The use case exercise gives you the reusable pattern
for applying them to a business question. CoWork makes that enriched
data accessible to authorized stakeholders without SQL. CoCo speeds up
building the next workload — and the one after that. Start with one,
and the rest follow naturally.

> Docs: [Cortex AI features overview](https://docs.snowflake.com/en/guides-overview-ai-features)
· [Snowflake CoCo](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code)

> **Apply on your account**
>
> Adapt the three prompts above to your own schema. Paste an error message
> you encounter, describe a query in
> plain English, or ask how a Cortex AI function behaves on edge-case
> inputs. The pattern is the same whether you are working in the Snowsight
> experience, CoCo Desktop, or a `cortex` session in your local
> terminal.

## Cleanup

Drop all objects created during the lab.

In [ ]:
-- Remove the /* and */ block comments when you are ready to clean up the lab objects.
-- Leaving commented out by default prevents accidental deletion when running all cells.
/*
USE ROLE SYSADMIN;
DROP DATABASE IF EXISTS AI_WITHOUT_THE_PHD_CORTEX_AI_HOL;
USE ROLE SYSADMIN;
DROP WAREHOUSE IF EXISTS AI_WITHOUT_THE_PHD_CORTEX_AI_WH;
*/